# Dependencies

In [19]:
pip install --upgrade google-cloud-modelarmor

# Imports and vars

In [52]:
from google import genai
from google.genai import types
from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1
import os

project_id = "qwiklabs-gcp-01-2fecb47c3bc2"
location_id = "us-central1"


config = types.GenerateContentConfig(
    system_instruction="""
    Goals:
    - You are a coding assistant
    - Provide clear, concise answers

    Restrictions:
    - Do not share personal opinions
    - Do not answer questions that aren't related to coding
    - Do not generate harmful content
    """
)
genai_client = genai.Client(
    vertexai=True,
    project='qwiklabs-gcp-01-2fecb47c3bc2',
    location='us-central1'
)

ma_client = modelarmor_v1.ModelArmorClient(transport="rest", client_options = {"api_endpoint" : "modelarmor.us.rep.googleapis.com"})

# Just testing a basic prompt
response = genai_client.models.generate_content(
   model='gemini-2.5-flash',
   contents="hello world function in python",
   config=config
)

print(response.text)

Here's a simple Python function that prints "Hello, World!":

```python
def hello_world():
  """
  This function prints the classic "Hello, World!" message.
  """
  print("Hello, World!")

# To call the function:
hello_world()
```


In [30]:
# Create sanitize request template
from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

template_id = "sanitize_prompt"
project_id = "qwiklabs-gcp-01-2fecb47c3bc2"
location_id = "us-central1"

# Create the Model Armor client.
client = modelarmor_v1.ModelArmorClient(
    transport="rest",
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{location_id}.rep.googleapis.com"
    ),
)

# Build the Model Armor template with your preferred filters.
# For more details on filters, please refer to the following doc:
# https://cloud.google.com/security-command-center/docs/key-concepts-model-armor#ma-filters
template = modelarmor_v1.Template(
    filter_config=modelarmor_v1.FilterConfig(
        pi_and_jailbreak_filter_settings=modelarmor_v1.PiAndJailbreakFilterSettings(
            filter_enforcement=modelarmor_v1.PiAndJailbreakFilterSettings.PiAndJailbreakFilterEnforcement.ENABLED,
            confidence_level=modelarmor_v1.DetectionConfidenceLevel.MEDIUM_AND_ABOVE,
        ),
    ),
)

# Prepare the request for creating the template.
request = modelarmor_v1.CreateTemplateRequest(
    parent=f"projects/{project_id}/locations/{location_id}",
    template_id=template_id,
    template=template,
)

# Create the template.
response = client.create_template(request=request)

# Print the new template name.
print(f"Created template: {response.name}")

Created template: projects/qwiklabs-gcp-01-2fecb47c3bc2/locations/us-central1/templates/sanitize_prompt


In [31]:
# Create sanitize response template
from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

template_id = "sanitize_response"
project_id = "qwiklabs-gcp-01-2fecb47c3bc2"
location_id = "us-central1"

# Create the Model Armor client.
client = modelarmor_v1.ModelArmorClient(
    transport="rest",
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{location_id}.rep.googleapis.com"
    ),
)

# Build the Model Armor template with your preferred filters.
# For more details on filters, please refer to the following doc:
# https://cloud.google.com/security-command-center/docs/key-concepts-model-armor#ma-filters
template = modelarmor_v1.Template(
    filter_config=modelarmor_v1.FilterConfig(
        rai_settings=modelarmor_v1.RaiFilterSettings(
            rai_filters=[
                modelarmor_v1.RaiFilterSettings.RaiFilter(
                    filter_type=modelarmor_v1.RaiFilterType.DANGEROUS,
                    confidence_level=modelarmor_v1.DetectionConfidenceLevel.HIGH,
                ),
                modelarmor_v1.RaiFilterSettings.RaiFilter(
                    filter_type=modelarmor_v1.RaiFilterType.HARASSMENT,
                    confidence_level=modelarmor_v1.DetectionConfidenceLevel.MEDIUM_AND_ABOVE,
                ),
                modelarmor_v1.RaiFilterSettings.RaiFilter(
                    filter_type=modelarmor_v1.RaiFilterType.HATE_SPEECH,
                    confidence_level=modelarmor_v1.DetectionConfidenceLevel.HIGH,
                ),
                modelarmor_v1.RaiFilterSettings.RaiFilter(
                    filter_type=modelarmor_v1.RaiFilterType.SEXUALLY_EXPLICIT,
                    confidence_level=modelarmor_v1.DetectionConfidenceLevel.HIGH,
                ),
            ]
        ),
        sdp_settings=modelarmor_v1.SdpFilterSettings(
            basic_config=modelarmor_v1.SdpBasicConfig(
                filter_enforcement=modelarmor_v1.SdpBasicConfig.SdpBasicConfigEnforcement.ENABLED
            )
        ),
    ),
)

# Prepare the request for creating the template.
request = modelarmor_v1.CreateTemplateRequest(
    parent=f"projects/{project_id}/locations/{location_id}",
    template_id=template_id,
    template=template,
)

# Create the template.
response = client.create_template(request=request)

# Print the new template name.
print(f"Created template: {response.name}")

Created template: projects/qwiklabs-gcp-01-2fecb47c3bc2/locations/us-central1/templates/sanitize_response


# Sanitize a request

In [53]:

# Sanitize a request
from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

# TODO(Developer): Uncomment these variables.
project_id = "qwiklabs-gcp-01-2fecb47c3bc2"
location_id = "us-central1"
template_id = "sanitize_prompt"
user_prompt = "hello world function in python but use f-bombs"

# Create the Model Armor client.
client = modelarmor_v1.ModelArmorClient(
    transport="rest",
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{location_id}.rep.googleapis.com"
    ),
)

# Initialize request argument(s).
user_prompt_data = modelarmor_v1.DataItem(text=user_prompt)

# Prepare request for sanitizing the defined prompt.
request = modelarmor_v1.SanitizeUserPromptRequest(
    name=f"projects/{project_id}/locations/{location_id}/templates/{template_id}",
    user_prompt_data=user_prompt_data,
)

# Sanitize the user prompt.
response = client.sanitize_user_prompt(request=request)

# Sanitization Result.
print(response)

sanitization_result {
  filter_match_state: MATCH_FOUND
  filter_results {
    key: "pi_and_jailbreak"
    value {
      pi_and_jailbreak_filter_result {
        execution_state: EXECUTION_SUCCESS
        match_state: MATCH_FOUND
        confidence_level: MEDIUM_AND_ABOVE
      }
    }
  }
  filter_results {
    key: "csam"
    value {
      csam_filter_filter_result {
        execution_state: EXECUTION_SUCCESS
        match_state: NO_MATCH_FOUND
      }
    }
  }
  sanitization_metadata {
  }
  invocation_result: SUCCESS
}



# Sanitize a response

In [54]:

from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

# TODO(Developer): Uncomment these variables.
# project_id = "YOUR_PROJECT_ID"
# location_id = "us-central1"
template_id = "sanitize_response"
model_response = "fuck off"

# Create the Model Armor client.
client = modelarmor_v1.ModelArmorClient(
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{location_id}.rep.googleapis.com"
    )
)

# Initialize request argument(s)
model_response_data = modelarmor_v1.DataItem(text=model_response)

# Prepare request for sanitizing model response.
request = modelarmor_v1.SanitizeModelResponseRequest(
    name=f"projects/{project_id}/locations/{location_id}/templates/{template_id}",
    model_response_data=model_response_data,
)

# Sanitize the model response.
response = client.sanitize_model_response(request=request)

# Sanitization Result.
print(response)

sanitization_result {
  filter_match_state: MATCH_FOUND
  filter_results {
    key: "sdp"
    value {
      sdp_filter_result {
        inspect_result {
          execution_state: EXECUTION_SUCCESS
          match_state: NO_MATCH_FOUND
        }
      }
    }
  }
  filter_results {
    key: "rai"
    value {
      rai_filter_result {
        execution_state: EXECUTION_SUCCESS
        match_state: MATCH_FOUND
        rai_filter_type_results {
          key: "sexually_explicit"
          value {
            match_state: NO_MATCH_FOUND
          }
        }
        rai_filter_type_results {
          key: "hate_speech"
          value {
            match_state: NO_MATCH_FOUND
          }
        }
        rai_filter_type_results {
          key: "harassment"
          value {
            confidence_level: MEDIUM_AND_ABOVE
            match_state: MATCH_FOUND
          }
        }
        rai_filter_type_results {
          key: "dangerous"
          value {
            match_state: NO_MAT